# Project 9: Optimizer Comparison Using a Neural Network

## Introduction

- An **optimizer** is an algorithm that updates the model's weights during training to reduce the loss
- Different optimizers update the weights in different ways — some are faster, some are more stable
- In this project, two commonly used optimizers are compared: **SGD (Stochastic Gradient Descent)** and **Adam**
- Both are trained on the **Iris dataset** and their training loss and accuracy are compared

## Task 1: Build a Simple Neural Network Using the Iris Dataset

- Load the Iris dataset (3 classes: Setosa, Versicolor, Virginica)
- Normalize the features and one-hot encode the labels
- Define the neural network structure:
  - Input layer: 4 neurons
  - Hidden layer: 8 neurons (ReLU activation)
  - Output layer: 3 neurons (Softmax activation)

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

In [3]:
# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [4]:
# One-hot encode labels
y_encoded = np.eye(3)[y]

In [5]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42
)

In [6]:
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Network structure: 4 -> 8 -> 3")

Training samples: 120
Testing samples: 30
Network structure: 4 -> 8 -> 3


In [7]:
# Activation functions
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [8]:
# Loss and accuracy
def cross_entropy_loss(y, y_pred):
    y_pred = np.clip(y_pred, 1e-10, 1.0)
    return -np.mean(np.sum(y * np.log(y_pred), axis=1))

def accuracy(y, y_pred):
    return np.mean(np.argmax(y, axis=1) == np.argmax(y_pred, axis=1)) * 100

## Task 2: Train the Model Using SGD and Adam Optimizers

### SGD (Stochastic Gradient Descent)
- Updates weights using the gradient of the loss directly
- Uses a fixed learning rate
- Formula: `W = W - lr * gradient`

### Adam Optimizer
- Adapts the learning rate for each parameter automatically
- Uses two moving averages — mean of gradients and mean of squared gradients
- Generally converges faster than SGD

In [9]:
def init_params():
    np.random.seed(42)
    W1 = np.random.randn(4, 8) * 0.01
    b1 = np.zeros((1, 8))
    W2 = np.random.randn(8, 3) * 0.01
    b2 = np.zeros((1, 3))
    return W1, b1, W2, b2

In [10]:
def forward(X, W1, b1, W2, b2):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)
    return Z1, A1, A2

def backward(X, y, Z1, A1, A2, W2):
    n   = len(y)
    dZ2 = A2 - y
    dW2 = np.dot(A1.T, dZ2) / n
    db2 = np.mean(dZ2, axis=0, keepdims=True)
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / n
    db1 = np.mean(dZ1, axis=0, keepdims=True)
    return dW1, db1, dW2, db2

In [11]:
# ── Train with SGD ──────────────────────────────────────────
def train_sgd(X_train, y_train, epochs=100, lr=0.1):
    W1, b1, W2, b2 = init_params()
    loss_history = []
    for epoch in range(epochs):
        Z1, A1, A2 = forward(X_train, W1, b1, W2, b2)
        loss = cross_entropy_loss(y_train, A2)
        loss_history.append(loss)
        dW1, db1, dW2, db2 = backward(X_train, y_train, Z1, A1, A2, W2)
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2
    return W1, b1, W2, b2, loss_history

In [12]:
# ── Train with Adam ─────────────────────────────────────────
def train_adam(X_train, y_train, epochs=100, lr=0.01):
    W1, b1, W2, b2 = init_params()
    loss_history = []
    beta1, beta2, epsilon = 0.9, 0.999, 1e-8
    mW1=mb1=mW2=mb2=vW1=vb1=vW2=vb2 = 0

    for epoch in range(1, epochs + 1):
        Z1, A1, A2 = forward(X_train, W1, b1, W2, b2)
        loss = cross_entropy_loss(y_train, A2)
        loss_history.append(loss)
        dW1, db1, dW2, db2 = backward(X_train, y_train, Z1, A1, A2, W2)

        mW1 = beta1*mW1 + (1-beta1)*dW1
        vW1 = beta2*vW1 + (1-beta2)*dW1**2
        mW2 = beta1*mW2 + (1-beta1)*dW2
        vW2 = beta2*vW2 + (1-beta2)*dW2**2
        mb1 = beta1*mb1 + (1-beta1)*db1
        vb1 = beta2*vb1 + (1-beta2)*db1**2
        mb2 = beta1*mb2 + (1-beta1)*db2
        vb2 = beta2*vb2 + (1-beta2)*db2**2

        mW1_c = mW1/(1-beta1**epoch); vW1_c = vW1/(1-beta2**epoch)
        mW2_c = mW2/(1-beta1**epoch); vW2_c = vW2/(1-beta2**epoch)
        mb1_c = mb1/(1-beta1**epoch); vb1_c = vb1/(1-beta2**epoch)
        mb2_c = mb2/(1-beta1**epoch); vb2_c = vb2/(1-beta2**epoch)

        W1 -= lr * mW1_c / (np.sqrt(vW1_c) + epsilon)
        b1 -= lr * mb1_c / (np.sqrt(vb1_c) + epsilon)
        W2 -= lr * mW2_c / (np.sqrt(vW2_c) + epsilon)
        b2 -= lr * mb2_c / (np.sqrt(vb2_c) + epsilon)

    return W1, b1, W2, b2, loss_history

In [13]:
# Run both optimizers
W1_sgd,  b1_sgd,  W2_sgd,  b2_sgd,  loss_sgd  = train_sgd(X_train, y_train)
W1_adam, b1_adam, W2_adam, b2_adam, loss_adam = train_adam(X_train, y_train)

print("SGD  — Initial Loss:", round(loss_sgd[0], 4),  "| Final Loss:", round(loss_sgd[-1], 4))
print("Adam — Initial Loss:", round(loss_adam[0], 4), "| Final Loss:", round(loss_adam[-1], 4))

SGD  — Initial Loss: 1.0987 | Final Loss: 0.6473
Adam — Initial Loss: 1.0987 | Final Loss: 0.0951


## Task 3: Compare Training Results and Classification Performance

In [14]:
# Loss at key checkpoints
print(f"{'Epoch':<10} {'SGD Loss':<15} {'Adam Loss'}")
print("-" * 35)
checkpoints = [0, 9, 19, 39, 59, 79, 99]
for i in checkpoints:
    print(f"{i+1:<10} {round(loss_sgd[i],4):<15} {round(loss_adam[i],4)}")

Epoch      SGD Loss        Adam Loss
-----------------------------------
1          1.0987          1.0987
10         1.0984          1.0327
20         1.0982          0.7737
40         1.0958          0.3764
60         1.0718          0.235
80         0.8946          0.142
100        0.6473          0.0951


In [15]:
# Test accuracy
_, _, A2_sgd  = forward(X_test, W1_sgd,  b1_sgd,  W2_sgd,  b2_sgd)
_, _, A2_adam = forward(X_test, W1_adam, b1_adam, W2_adam, b2_adam)

acc_sgd  = accuracy(y_test, A2_sgd)
acc_adam = accuracy(y_test, A2_adam)

print("── Classification Performance on Test Set ──\n")
print(f"SGD  Accuracy : {round(acc_sgd, 2)}%")
print(f"Adam Accuracy : {round(acc_adam, 2)}%")

── Classification Performance on Test Set ──

SGD  Accuracy : 80.0%
Adam Accuracy : 100.0%


## Conclusion

- Both SGD and Adam were trained for 100 epochs on the Iris dataset and their loss and accuracy were compared
- **Adam** reduced the loss faster than SGD — it adapts the learning rate automatically for each parameter, which helps it converge quicker
- **SGD** is simpler and works well but may need more epochs or careful tuning of the learning rate to perform as well as Adam
- In terms of accuracy on the test set, Adam generally performs better or equal to SGD in fewer iterations
- For most deep learning problems, **Adam is the preferred choice** as a starting optimizer because of its speed and reliability
- SGD is still useful when fine control over training is needed or for very large models